# Agent Skills – das Prinzip, bewusst einfach

**Lernziel:** Sie verstehen drei Dinge:

1. Ein Skill besitzt Metadaten (`name`, `description`) und Anweisungen in `SKILL.md`.
2. Zunächst wird nur ein kleiner **Katalog** der verfügbaren Skills bekannt gemacht.
3. Erst nach der Auswahl wird der vollständige Anweisungsteil des passenden Skills genutzt.

Am Ende vergleichen wir dieselbe Aufgabe **mit** und **ohne** Skill.

> **Wichtig:** Dieses Notebook bildet das Prinzip didaktisch nach. Wie eine konkrete Agent-Runtime Skills entdeckt und aktiviert, ist implementierungsabhängig. Der offene Agent-Skills-Standard legt das Dateiformat und das Prinzip der Progressive Disclosure fest, nicht einen bestimmten Routing-Algorithmus.


## 1. Setup

Für die Unterrichtsstunde verwenden wir **nur einen** OpenAI-kompatiblen lokalen Endpunkt (z. B. LM Studio). Dadurch entfällt die Backend-Auswahl aus der früheren Fassung.

Die Funktion `chat()` kapselt den Modellaufruf. Für das Skill-Prinzip ist ihre interne API-Syntax nicht wichtig.


In [ ]:
# Einmalig installieren, falls nötig:
# %pip install openai pyyaml

from pathlib import Path
import yaml
from openai import OpenAI

MODEL = "prism-ml/bonsai-27b"   # an das lokal geladene Modell anpassen

client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

def chat(system, user):
    antwort = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return antwort.choices[0].message.content.strip()


## 2. Zwei kleine Skills anlegen

Ein Skill ist ein Verzeichnis mit einer `SKILL.md`. Im offenen Format sind `name` und `description` Pflichtfelder.

Die Regeln sind absichtlich ungewöhnlich. So können wir später sehen, ob der Skill tatsächlich einen Unterschied gemacht hat.


In [ ]:
kaffee = """---
name: kaffee
description: Zubereitung von Kaffee nach Hausstandard. Verwenden bei Kaffee, Espresso, Filterkaffee oder einem gewünschten Wachmacher. Nicht verwenden für Tee.
---

# Kaffee zubereiten

## Hausregeln
1. Mengen nur in Gramm angeben, niemals in Löffeln.
2. Genau drei nummerierte Schritte schreiben.
3. Eine Temperatur mit Toleranz angeben, z. B. 94 °C ± 2.
4. Die letzte Zeile beginnt mit: Häufigster Fehler:

## Fachliche Vorgaben
- Filterkaffee: Verhältnis 1:16.
- Mahlgrad: mittel.
- Brühzeit: 3 bis 4 Minuten.
"""

tee = """---
name: tee
description: Zubereitung von Tee nach Hausstandard. Verwenden bei Tee, Grüntee, Schwarztee, Kräutertee oder einem beruhigenden Heißgetränk. Nicht verwenden für Kaffee.
---

# Tee zubereiten

## Hausregeln
1. Mengen nur in Gramm angeben, niemals in Löffeln.
2. Genau drei nummerierte Schritte schreiben.
3. Eine Temperatur mit Toleranz angeben, z. B. 80 °C ± 2.
4. Die letzte Zeile beginnt mit: Häufigster Fehler:

## Fachliche Vorgaben
- Grüntee: 80 °C, 2 Minuten.
- Schwarztee: 95 °C, 3 Minuten.
- Kräutertee: 100 °C, 8 Minuten.
"""

Path("skills/kaffee").mkdir(parents=True, exist_ok=True)
Path("skills/tee").mkdir(parents=True, exist_ok=True)

Path("skills/kaffee/SKILL.md").write_text(kaffee, encoding="utf-8")
Path("skills/tee/SKILL.md").write_text(tee, encoding="utf-8")

print("Skill-Dateien angelegt.")


## 3. Ebene 1: nur den Katalog aufbauen

Für die **Discovery** brauchen wir zunächst nur `name` und `description`.

Wir lesen das YAML-Frontmatter mit `yaml.safe_load()`. Dadurch entfällt der handgeschriebene YAML-Parser der früheren Fassung.


In [ ]:
katalog = ""
namen = []

for pfad in Path("skills").glob("*/SKILL.md"):
    text = pfad.read_text(encoding="utf-8")
    frontmatter = text.split("---", 2)[1]
    meta = yaml.safe_load(frontmatter)

    namen.append(meta["name"])
    katalog += "- " + meta["name"] + ": " + meta["description"] + "\n"

print(katalog)


Der Katalog enthält nur die Metadaten. Die vollständigen Anweisungen der Skills werden dem Modell noch **nicht** gegeben. Genau das ist die erste Stufe der Progressive Disclosure.


## 4. Einen passenden Skill auswählen

Hier simulieren wir die Aktivierung mit einem Modellaufruf. Das ist **eine didaktische Implementierung**, nicht die vorgeschriebene interne Architektur einer Agent-Runtime.

Eine andere Runtime könnte die Aktivierung anders lösen oder einen Skill explizit durch den Nutzer auswählen lassen.


In [ ]:
WUNSCH = "Ich brauche etwas, das mich wach macht."

router_prompt = """Wähle den passenden Skill aus diesem Katalog:

{0}

Antworte nur mit dem Skill-Namen.
Wenn kein Skill passt, antworte: keiner
""".format(katalog)

name = chat(router_prompt, WUNSCH).lower().strip()

if name not in namen:
    name = None

print("Gewählter Skill:", name)


## 5. Ebene 2: erst jetzt die Anweisungen laden

Nur wenn ein Skill gewählt wurde, lesen wir seinen Markdown-Rumpf und verwenden ihn als zusätzliche Anweisung für die Aufgabe.


In [ ]:
if name is None:
    print("Kein Skill passt.")
    mit_skill = None
else:
    pfad = Path("skills") / name / "SKILL.md"
    text = pfad.read_text(encoding="utf-8")
    anweisungen = text.split("---", 2)[2].strip()

    mit_skill = chat(anweisungen, WUNSCH)
    print(mit_skill)


## 6. Kontrollversuch: dieselbe Aufgabe ohne Skill

Jetzt bekommt dasselbe Modell denselben Wunsch, aber **nicht** die Hausregeln aus `SKILL.md`.

So testen wir nicht, ob das Modell allgemein Kaffee kennt, sondern ob der Skill einen beobachtbaren Unterschied erzeugt.


In [ ]:
ohne_skill = chat(
    "Beantworte die Anfrage hilfreich und knapp.",
    WUNSCH
)

print(ohne_skill)


## 7. Die Hausregeln einfach prüfen

Für die Einführung genügt ein kleiner deterministischer Prüfschritt. Wir verzichten bewusst auf reguläre Ausdrücke, `subprocess` und ein separat erzeugtes Python-Skript.

In einem echten Skill könnte eine solche Prüfung später als Datei unter `scripts/` liegen.


In [ ]:
def pruefe(titel, text):
    if not text:
        print(titel, ": keine Antwort")
        return

    schritte = 0
    for zeile in text.splitlines():
        if zeile.startswith(("1.", "2.", "3.", "4.")):
            schritte += 1

    letzte_zeile = text.strip().splitlines()[-1]

    print("\n" + titel)
    print("Gramm statt Löffel:      ", " g" in text and "Löffel" not in text)
    print("Genau drei Schritte:     ", schritte == 3)
    print("Temperatur mit Toleranz: ", "°C ±" in text)
    print("Fehlersatz am Ende:      ", letzte_zeile.startswith("Häufigster Fehler:"))

pruefe("MIT SKILL", mit_skill)
pruefe("OHNE SKILL", ohne_skill)


## 8. Was wurde technisch gezeigt?

```text
Nutzerwunsch
    ↓
Katalog aus name + description
    ↓
passenden Skill auswählen
    ↓
SKILL.md dieses Skills laden
    ↓
Aufgabe mit den zusätzlichen Anweisungen bearbeiten
```

Damit sind die beiden wichtigsten Ebenen sichtbar:

- **Ebene 1 – Metadaten:** klein und für die Auswahl verfügbar.
- **Ebene 2 – Anweisungen:** erst nach Aktivierung in den Modellkontext.

Die optionale **Ebene 3** besteht aus weiteren Ressourcen wie `scripts/`, `references/` oder `assets/`, die nur bei Bedarf genutzt werden.


## Aufgaben

1. Ändern Sie `WUNSCH` zu „Ich möchte etwas Beruhigendes für den Abend.“ Welcher Skill wird gewählt?
2. Testen Sie „Erkläre mir, wie Koffein im Körper wirkt.“ Warum sollte hier **kein** Zubereitungs-Skill greifen?
3. Verschlechtern Sie die `description` eines Skills absichtlich und beobachten Sie die Auswahl.
4. Legen Sie einen dritten Skill `kakao` an. Der Python-Code für Katalog und Auswahl soll unverändert bleiben.
5. Überführen Sie die Funktion `pruefe()` als Erweiterungsaufgabe in `scripts/pruefe_hausregeln.py`.

### Merksätze

- `description` hilft dem Agenten zu entscheiden, **wann** ein Skill relevant ist.
- Der Markdown-Rumpf beschreibt, **wie** die Aufgabe ausgeführt werden soll.
- Progressive Disclosure bedeutet: erst Metadaten, dann Anweisungen, dann optionale Ressourcen.
- Eine Modellentscheidung zur Aktivierung kann fehlschlagen und sollte getestet werden.
- Deterministische Prüfungen sind sinnvoll, wenn ein Ergebnis eindeutig maschinell überprüfbar ist.


## Technische Grundlage

Diese Notebook-Fassung orientiert sich am offenen Agent-Skills-Standard:

- Spezifikation: https://agentskills.io/specification
- Client-Implementierung / Progressive Disclosure: https://agentskills.io/client-implementation/adding-skills-support

Der Standard definiert `SKILL.md`, die Pflichtfelder `name` und `description`, optionale Ressourcenverzeichnisse und das Prinzip der gestuften Bereitstellung. Die konkrete Aktivierungslogik bleibt Sache der jeweiligen Agent-Runtime.


# Fortgeschrittene Übung: Tool-Use mit einem Skill-Skript

Bis hierhin hat der Skill nur **Kontext und Anweisungen** geliefert.

Jetzt kommt eine dritte Ebene hinzu: Der Skill bringt ein kleines Python-Skript mit. Dieses Skript prüft die erzeugte Antwort deterministisch.

> **Wichtig:** Auch dieser Abschnitt ist eine didaktische Simulation. Der offene Agent-Skills-Standard legt fest, dass Skills Skripte enthalten können. Wie eine konkrete Agent-Runtime solche Skripte aufruft, ist jedoch runtimeabhängig.


## 9. Ein Prüfskript zum Skill hinzufügen

Das Skript liegt im Skill-Verzeichnis unter `scripts/`.

Es bekommt den Pfad zu einer Textdatei übergeben und prüft dieselben Hausregeln wie zuvor die Funktion `pruefe()`. Der Unterschied: Die Prüfung ist jetzt **aus dem Notebook ausgelagert** und Bestandteil des Skills.


In [ ]:
pruefskript = r"""
from pathlib import Path
import sys

text = Path(sys.argv[1]).read_text(encoding="utf-8")

schritte = 0
for zeile in text.splitlines():
    if zeile.startswith(("1.", "2.", "3.", "4.")):
        schritte += 1

letzte_zeile = text.strip().splitlines()[-1]

print("Gramm statt Löffel:", " g" in text and "Löffel" not in text)
print("Genau drei Schritte:", schritte == 3)
print("Temperatur mit Toleranz:", "°C ±" in text)
print("Fehlersatz am Ende:", letzte_zeile.startswith("Häufigster Fehler:"))
"""

for skill_name in ["kaffee", "tee"]:
    scripts = Path("skills") / skill_name / "scripts"
    scripts.mkdir(parents=True, exist_ok=True)

    pfad = scripts / "pruefe_hausregeln.py"
    pfad.write_text(pruefskript, encoding="utf-8")

print("Prüfskripte angelegt.")


## 10. Den Skill auf das Tool verweisen lassen

Ein Skript allein reicht nicht. Die Anweisungen müssen erklären,

- **wann** das Skript verwendet wird,
- **welche Eingabe** es erwartet,
- und **was mit dem Ergebnis** geschehen soll.

Wir ergänzen deshalb beide `SKILL.md`-Dateien um einen kurzen Tool-Hinweis.


In [ ]:
tool_hinweis = """

## Prüfung
Nach dem Erstellen der Antwort:
1. Speichere die Antwort als Textdatei.
2. Führe `scripts/pruefe_hausregeln.py` mit dieser Datei als Argument aus.
3. Wenn eine Prüfung `False` meldet, überarbeite die Antwort.
"""

for skill_name in ["kaffee", "tee"]:
    pfad = Path("skills") / skill_name / "SKILL.md"
    text = pfad.read_text(encoding="utf-8")

    if "## Prüfung" not in text:
        pfad.write_text(text + tool_hinweis, encoding="utf-8")

print("Tool-Hinweis ergänzt.")


## 11. Das Tool ausführen

Für diese Übung übernimmt das Notebook die Rolle der Runtime:

1. Es speichert die Modellantwort.
2. Es bestimmt das Skript des **tatsächlich gewählten Skills**.
3. Es führt das Skript aus.
4. Es zeigt dessen Ausgabe an.

Dadurch bleibt sichtbar, welche Aufgabe das Modell übernimmt und welche Aufgabe deterministisch geprüft wird.


In [ ]:
import subprocess
import sys

if name is None or mit_skill is None:
    print("Kein Skill aktiv – daher kein Skill-Tool.")
else:
    antwort_datei = Path("antwort.txt")
    antwort_datei.write_text(mit_skill, encoding="utf-8")

    skript = Path("skills") / name / "scripts" / "pruefe_hausregeln.py"

    ergebnis = subprocess.run(
        [sys.executable, str(skript), str(antwort_datei)],
        capture_output=True,
        text=True
    )

    print(ergebnis.stdout)


## 12. Was ist hier der eigentliche Tool-Use?

Der Ablauf ist jetzt:

```text
Nutzerwunsch
    ↓
Skill auswählen
    ↓
SKILL.md laden
    ↓
Modell erzeugt eine Antwort
    ↓
Skill-Skript wird ausgeführt
    ↓
deterministisches Prüfergebnis
```

Das Skript ist damit kein zusätzlicher Prompt, sondern ein **ausführbares Werkzeug**.

Die Trennung ist bewusst:

- Das Modell formuliert und entscheidet sprachlich.
- Das Skript prüft eine eindeutig maschinell kontrollierbare Eigenschaft.
- Die Runtime verbindet beides.

In produktiven Agent-Systemen kann die Tool-Ausführung automatisch erfolgen. Dieses Notebook führt den Ablauf explizit aus, damit jede Stufe nachvollziehbar bleibt.


## 13. Fortgeschrittene Aufgaben

1. Verändern Sie die Antwort absichtlich so, dass eine Hausregel verletzt wird. Erkennt das Skript den Fehler?
2. Ergänzen Sie im Prüfsript eine fünfte Regel.
3. Ändern Sie `WUNSCH`, sodass der Tee-Skill gewählt wird. Prüfen Sie, ob nun das Skript aus `skills/tee/scripts/` ausgeführt wird.
4. Überlegen Sie: Welche Regeln eignen sich **nicht** für ein deterministisches Skript?
5. Optional: Lassen Sie das Notebook bei einem `False` automatisch eine zweite Modellantwort erzeugen. Begrenzen Sie die Zahl der Versuche, damit keine Endlosschleife entsteht.

### Merksatz

Ein Skill kann **Anweisungen und Werkzeuge** bündeln. Das Modell muss nicht jede Teilaufgabe selbst lösen: Wiederkehrende, eindeutig prüfbare Schritte können an deterministischen Code delegiert werden.


# Ausbaustufe 3: Websuche über MCP

In den ersten beiden Stufen haben wir zwei unterschiedliche Erweiterungen gesehen:

```text
Stufe 1: Skill → zusätzliche Anweisungen und Fachkontext
Stufe 2: Skill → lokales Skript für eine deterministische Prüfung
```

Jetzt kommt eine externe Fähigkeit hinzu:

```text
Stufe 3: Skill → entscheidet über Vorgehen
                 ↓
                MCP
                 ↓
             Websuche
                 ↓
        externe Informationen
                 ↓
        Antwort nach Skill-Regeln
```

Die zentrale Idee:

> **Der Skill beschreibt, wann und wie recherchiert werden soll. MCP stellt die externe Fähigkeit bereit.**

Für die Demo verwenden wir den gehosteten **Exa MCP Server**, weil er ohne lokale Serverinstallation erreichbar ist und für einfache Nutzung auch anonym mit Rate Limits funktioniert.

**Wichtig:** MCP ist hier nicht die Suchmaschine selbst. MCP ist die standardisierte Schnittstelle, über die unser Notebook das Suchwerkzeug entdeckt und aufruft.


## 14. MCP-Client installieren

Wir verwenden das offizielle Python-SDK für das Model Context Protocol.

```python
%pip install -U mcp
```

Die aktuelle SDK-Generation bietet einen sehr einfachen Client: Eine MCP-URL genügt, um über Streamable HTTP eine Verbindung aufzubauen.

> Falls die Installation in einer laufenden Jupyter-Sitzung erstmals erfolgt, kann ein Kernel-Neustart nötig sein.


In [ ]:
# Einmalig ausführen, falls das MCP-SDK noch nicht installiert ist:
# %pip install -U mcp

from mcp import Client
from mcp.types import TextContent

EXA_MCP_URL = "https://mcp.exa.ai/mcp"


## 15. Erst Discovery: Welche Tools bietet der Server an?

Bevor wir ein Tool aufrufen, fragen wir den MCP-Server nach seinen Fähigkeiten.

Das ist ein wichtiger Unterschied zu einer fest verdrahteten REST-API: Ein MCP-Client kann die angebotenen Tools mitsamt Beschreibung und Eingabeschema zur Laufzeit entdecken.


In [ ]:
async with Client(EXA_MCP_URL) as mcp:
    tools = await mcp.list_tools()

    for tool in tools.tools:
        print(tool.name, "–", tool.description)


Beim gehosteten Exa-Server gehören standardmäßig insbesondere diese Werkzeuge dazu:

- `web_search_exa` – Websuche mit aufbereitetem Inhalt
- `web_fetch_exa` – Inhalt konkreter Webseiten abrufen

Für unsere Einführung benötigen wir nur `web_search_exa`.


## 16. Eine möglichst kleine MCP-Websuche kapseln

Die Hilfsfunktion unten tut nur drei Dinge:

1. Verbindung zum MCP-Server öffnen,
2. `web_search_exa` aufrufen,
3. Textblöcke aus dem Tool-Ergebnis zurückgeben.

Damit bleibt die Protokollmechanik aus dem restlichen Notebook heraus.


In [ ]:
async def websuche(query, anzahl=5):
    async with Client(EXA_MCP_URL) as mcp:
        ergebnis = await mcp.call_tool(
            "web_search_exa",
            {
                "query": query,
                "numResults": anzahl
            }
        )

    if ergebnis.is_error:
        return "MCP-Suche fehlgeschlagen."

    texte = []
    for block in ergebnis.content:
        if isinstance(block, TextContent):
            texte.append(block.text)

    return "\n".join(texte)


## 17. Den Skill um eine Recherche-Regel ergänzen

Der Skill soll nicht bei jeder Kaffee- oder Tee-Anfrage automatisch suchen.

Er bekommt deshalb eine einfache Regel:

```text
Nutze externe Websuche nur dann, wenn
- der Nutzer ausdrücklich nach aktuellen oder extern belegten Empfehlungen fragt,
- eine Hausregel mit externen Empfehlungen verglichen werden soll,
- oder Informationen benötigt werden, die nicht im Skill stehen.

Unterscheide in der Antwort klar zwischen Hausregel und externer Empfehlung.
```

Damit bleibt die **Entscheidungslogik im Skill**, während die eigentliche Suchfähigkeit über MCP bereitgestellt wird.


In [ ]:
recherche_regel = """

## Externe Recherche
Nutze eine verfügbare Websuche nur dann, wenn
- der Nutzer ausdrücklich nach aktuellen oder extern belegten Empfehlungen fragt,
- eine Hausregel mit externen Empfehlungen verglichen werden soll,
- oder Informationen benötigt werden, die nicht in diesem Skill stehen.

Bei externer Recherche:
1. Nenne die verwendeten Quellen bzw. URLs.
2. Unterscheide klar zwischen Hausregel und externer Empfehlung.
3. Weise auf widersprüchliche Empfehlungen hin, statt eine Scheingenauigkeit zu erzeugen.
"""

for skill_name in ["kaffee", "tee"]:
    pfad = Path("skills") / skill_name / "SKILL.md"
    text = pfad.read_text(encoding="utf-8")

    if "## Externe Recherche" not in text:
        pfad.write_text(text + recherche_regel, encoding="utf-8")

print("Recherche-Regel ergänzt.")


## 18. Eine Anfrage mit echtem Informationsbedarf

Jetzt verwenden wir absichtlich eine Anfrage, bei der der lokale Skill allein nicht die ganze Antwort liefern soll:

> „Wie heiß sollte das Wasser für einen hell gerösteten Filterkaffee sein? Prüfe aktuelle externe Empfehlungen und vergleiche sie mit unserer Hausregel 94 °C ± 2.“

Die Hausregel kommt aus dem Skill.  
Die externe Vergleichsinformation kommt über MCP aus der Websuche.


In [ ]:
WEB_WUNSCH = (
    "Wie heiß sollte das Wasser für einen hell gerösteten Filterkaffee sein? "
    "Prüfe aktuelle externe Empfehlungen und vergleiche sie mit unserer "
    "Hausregel 94 °C ± 2."
)

# Den passenden Skill wieder über den bekannten Katalog bestimmen
web_name = chat(router_prompt, WEB_WUNSCH).lower().strip()

if web_name not in namen:
    web_name = None

print("Gewählter Skill:", web_name)


## 19. Die Suche über MCP ausführen

Für die erste MCP-Übung machen wir die Tool-Entscheidung bewusst **explizit**:

Der Nutzer verlangt eine externe Prüfung – deshalb rufen wir die Websuche auf.

In einer späteren produktiven Agent-Runtime könnte das Modell anhand der Skill-Anweisungen selbst entscheiden, wann das MCP-Tool benötigt wird.


In [ ]:
suchanfrage = (
    "light roast filter coffee brewing water temperature "
    "recommended temperature specialty coffee"
)

suchergebnis = await websuche(suchanfrage, anzahl=5)

print(suchergebnis[:6000])


## 20. Suchergebnis und Skill zusammenführen

Nun bekommt das Modell zwei unterschiedliche Kontextarten:

- **Skill:** unsere lokalen Regeln und die Arbeitsweise,
- **MCP-Websuche:** externe Informationen zur aktuellen Fragestellung.

Genau diese Trennung ist konzeptionell wichtig.


In [ ]:
if web_name is None:
    print("Kein passender Skill gefunden.")
else:
    pfad = Path("skills") / web_name / "SKILL.md"
    text = pfad.read_text(encoding="utf-8")
    anweisungen = text.split("---", 2)[2].strip()

    externe_quellen = suchergebnis[:6000]

    prompt = (
        WEB_WUNSCH
        + "\n\nHier sind Ergebnisse der externen Websuche:\n"
        + externe_quellen
        + "\n\nBeantworte die Anfrage nach den Regeln des Skills. "
          "Trenne Hausregel und externe Empfehlung deutlich."
    )

    antwort_mit_web = chat(anweisungen, prompt)
    print(antwort_mit_web)


## 21. Was wurde architektonisch ergänzt?

```text
                     ┌──────────────────────┐
                     │      SKILL.md        │
                     │ Regeln + Vorgehen    │
                     └──────────┬───────────┘
                                │
Nutzeranfrage ──→ Skill wählen ─┤
                                │
                                ▼
                    externe Information nötig?
                                │
                              ja│
                                ▼
                     ┌──────────────────────┐
                     │       MCP Client     │
                     └──────────┬───────────┘
                                │
                                ▼
                     ┌──────────────────────┐
                     │ Exa: web_search_exa │
                     └──────────┬───────────┘
                                │
                                ▼
                         Suchergebnisse
                                │
                                ▼
                         Modellantwort
```

### Die Rollen bleiben getrennt

| Bestandteil | Aufgabe |
|---|---|
| `SKILL.md` | beschreibt, **wie** die Aufgabe bearbeitet wird |
| MCP | standardisiert den Zugriff auf externe Fähigkeiten |
| Websuche | liefert externe, aktuelle Informationen |
| LLM | bewertet und formuliert die Antwort |
| lokales Skript aus Stufe 2 | prüft deterministische Regeln |

Damit wird sichtbar: **Skill, MCP und Tool sind keine konkurrierenden Konzepte. Sie ergänzen sich.**


## 22. Abgrenzung zu RAG

Gerade mit Ihrem RAG-Vorwissen lohnt sich der Vergleich:

| Bedarf | Naheliegender Mechanismus |
|---|---|
| „Unsere Hausregel ist 1:16.“ | Skill |
| „Was steht in unseren 500 internen Kaffee-Dokumenten?“ | RAG |
| „Was empfehlen aktuelle externe Quellen?“ | Websuche über MCP |
| „Berechne aus 37,5 g Kaffee exakt die Wassermenge bei 1:16.“ | deterministisches Tool/Skript |

Die Mechanismen können kombiniert werden. Ein Agent könnte beispielsweise zuerst interne Dokumente über RAG prüfen und nur bei fehlender Information zusätzlich eine externe Websuche ausführen.


## 23. Zwei weitere gut geeignete Search-MCPs

Für die Unterrichtsdemo wurde **Exa** gewählt, weil der gehostete Server besonders wenig Einrichtung benötigt.

### Brave Search MCP Server

Der offizielle Brave-Search-MCP-Server bietet u. a. `brave_web_search`, News-, Bild- und Videosuche. Er unterstützt HTTP und STDIO, benötigt für die Suche aber einen **Brave Search API Key**.

- Repository: https://github.com/brave/brave-search-mcp-server

### Tavily MCP

Tavily stellt einen gehosteten MCP-Server für Search, Extract, Map und Crawl bereit. Die Authentifizierung erfolgt über **API Key oder OAuth**. Dadurch ist er leistungsfähig, für die erste Demo aber etwas aufwendiger.

- Repository: https://github.com/tavily-ai/tavily-mcp

### Warum Exa für diese Übung?

- gehosteter MCP-Endpunkt,
- keine lokale Serverinstallation,
- anonyme Nutzung mit Rate Limits möglich,
- für die Demo reichen zwei Standardwerkzeuge.

Der Unterrichtspunkt ist nicht, einen bestimmten Suchanbieter zu lernen, sondern:

> **Ein MCP-Client kann eine externe Fähigkeit entdecken und über ein einheitliches Protokoll aufrufen.**


## 24. Fortgeschrittene Aufgaben

1. Ändern Sie die Nutzeranfrage zu einer normalen Zubereitungsfrage. **Soll überhaupt gesucht werden?**
2. Formulieren Sie eine Regel, anhand der das Modell selbst `SUCHE` oder `KEINE_SUCHE` entscheidet.
3. Lassen Sie anschließend nur bei `SUCHE` das MCP-Tool aufrufen.
4. Vergleichen Sie eine Kaffee-Empfehlung aus der Websuche mit einer internen Quelle aus einem vorhandenen RAG-System.
5. Diskutieren Sie: Welche externen Quellen würden Sie für Fachinformationen zulassen oder bevorzugen?
6. Welche zusätzlichen Guardrails wären nötig, wenn ein MCP-Tool nicht nur lesen, sondern Aktionen in einem externen System ausführen dürfte?

### Merksatz

> **Der Skill bestimmt die Arbeitsweise. MCP stellt Fähigkeiten standardisiert bereit. Die Runtime entscheidet, wann und mit welchen Rechten diese Fähigkeiten ausgeführt werden.**


## Technische Hinweise und Quellen

Stand: **11. September 2026**

- Exa MCP Server: https://github.com/exa-labs/exa-mcp-server  
  Hosted Endpoint: `https://mcp.exa.ai/mcp`
- Offizielles MCP Python SDK: https://github.com/modelcontextprotocol/python-sdk
- Model Context Protocol: https://modelcontextprotocol.io/
- Brave Search MCP Server: https://github.com/brave/brave-search-mcp-server
- Tavily MCP: https://github.com/tavily-ai/tavily-mcp

Der Exa-Endpunkt erlaubt laut aktueller Dokumentation anonyme Nutzung mit Rate Limits. Für höhere Limits oder zusätzliche Funktionen ist Authentifizierung erforderlich. Externe Dienste und deren Konditionen können sich ändern.
